In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load metric.csv
df_metric = pd.read_csv('metric.csv')
# Basic counts and distincts (limit lists to 50)
metric_row_count = int(df_metric.shape[0])
metric_distinct_cmdb = df_metric['cmdb_id'].dropna().astype(str).unique().tolist()[:50]
metric_distinct_kpis = df_metric['kpi_name'].dropna().astype(str).unique().tolist()[:50]
# Timestamp min/max converted to UTC+8 ISO strings
metric_ts_min = pd.to_datetime(df_metric['timestamp'].min(), unit='s', utc=True).tz_convert(tz).isoformat()
metric_ts_max = pd.to_datetime(df_metric['timestamp'].max(), unit='s', utc=True).tz_convert(tz).isoformat()
# Top 10 rows by timestamp ascending (compact dicts)
metric_top10 = df_metric.sort_values('timestamp', ascending=True).head(10)[['timestamp', 'cmdb_id', 'kpi_name', 'value']].to_dict(orient='records')

# Load trace.csv
df_trace = pd.read_csv('trace.csv')
trace_row_count = int(df_trace.shape[0])
trace_distinct_cmdb = df_trace['cmdb_id'].dropna().astype(str).unique().tolist()[:50]
trace_distinct_names = df_trace['trace_name'].dropna().astype(str).unique().tolist()[:50]
trace_ts_min = pd.to_datetime(df_trace['timestamp'].min(), unit='s', utc=True).tz_convert(tz).isoformat()
trace_ts_max = pd.to_datetime(df_trace['timestamp'].max(), unit='s', utc=True).tz_convert(tz).isoformat()
trace_top10 = df_trace.sort_values('timestamp', ascending=True).head(10)[['timestamp', 'cmdb_id', 'trace_name', 'value']].to_dict(orient='records')

# Load log.csv
df_log = pd.read_csv('log.csv')
log_row_count = int(df_log.shape[0])
log_distinct_cmdb = df_log['cmdb_id'].dropna().astype(str).unique().tolist()[:50]
log_distinct_names = df_log['log_name'].dropna().astype(str).unique().tolist()[:50]
log_ts_min = pd.to_datetime(df_log['timestamp'].min(), unit='s', utc=True).tz_convert(tz).isoformat()
log_ts_max = pd.to_datetime(df_log['timestamp'].max(), unit='s', utc=True).tz_convert(tz).isoformat()
log_top10 = df_log.sort_values('timestamp', ascending=True).head(10)[['timestamp', 'cmdb_id', 'log_name', 'value']].to_dict(orient='records')

# Load error_logs.csv
df_err = pd.read_csv('error_logs.csv')
err_row_count = int(df_err.shape[0])
err_distinct_cmdb = df_err['cmdb_id'].dropna().astype(str).unique().tolist()[:50]
err_ts_min = pd.to_datetime(df_err['timestamp'].min(), unit='s', utc=True).tz_convert(tz).isoformat()
err_ts_max = pd.to_datetime(df_err['timestamp'].max(), unit='s', utc=True).tz_convert(tz).isoformat()
err_earliest10 = df_err.sort_values('timestamp', ascending=True).head(10)[['timestamp', 'cmdb_id', 'message']].to_dict(orient='records')

# Assemble JSON-serializable summary
summary = {
    'metric.csv': {
        'row_count': metric_row_count,
        'distinct_cmdb_id': metric_distinct_cmdb,
        'distinct_kpi_name': metric_distinct_kpis,
        'timestamp_min_UTC+8': metric_ts_min,
        'timestamp_max_UTC+8': metric_ts_max,
        'top_10_rows': metric_top10
    },
    'trace.csv': {
        'row_count': trace_row_count,
        'distinct_cmdb_id': trace_distinct_cmdb,
        'distinct_trace_name': trace_distinct_names,
        'timestamp_min_UTC+8': trace_ts_min,
        'timestamp_max_UTC+8': trace_ts_max,
        'top_10_rows': trace_top10
    },
    'log.csv': {
        'row_count': log_row_count,
        'distinct_cmdb_id': log_distinct_cmdb,
        'distinct_log_name': log_distinct_names,
        'timestamp_min_UTC+8': log_ts_min,
        'timestamp_max_UTC+8': log_ts_max,
        'top_10_rows': log_top10
    },
    'error_logs.csv': {
        'row_count': err_row_count,
        'distinct_cmdb_id': err_distinct_cmdb,
        'timestamp_min_UTC+8': err_ts_min,
        'timestamp_max_UTC+8': err_ts_max,
        'earliest_10_rows': err_earliest10
    }
}

summary

```
Out[1]:
```
Summary of the loaded telemetry (plain English):

- metric.csv
  - Total rows: 321,463.
  - Distinct cmdb_id (sample, up to 50) includes many services, service pods, and nodes — e.g., adservice, adservice-0/1/2, frontend, frontend-0/1/2, paymentservice*, productcatalogservice*, recommendationservice*, cartservice*, shippingservice*, and node-1..node-6.
  - Distinct KPI names (sample, up to 50) include many JVM/runtime metrics and application RPC/HTTP metrics (e.g., app.grpc.*, app.http.*, runtime.java_lang_*, runtime.java_lang_GarbageCollector_*).
  - Timestamp range (UTC+8): 2022-03-21T01:00:00+08:00 through 2022-03-21T02:00:00+08:00.
  - Earliest/top rows (by timestamp) show adservice JVM GC and memory-related KPIs at timestamp 1647795600.

- trace.csv
  - Total rows: 29,328.
  - Distinct cmdb_id (sample) are primarily pod instances (e.g., adservice-0/1/2, cartservice-*, checkoutservice-*, frontend-*, paymentservice-*, etc.).
  - Distinct trace names (sample) include many trace-origin and trace-self metrics, e.g., trace.self.duration_mean, trace.from_frontend-*.duration_mean/p95/error_rate/row_count, trace.from_checkoutservice-*.*
  - Timestamp range (UTC+8): 2022-03-21T01:30:00+08:00 through 2022-03-21T01:59:00+08:00.
  - Earliest/top trace rows show per-pod trace metrics at timestamp 1647797400 (mostly zero or very small durations, with row_count values present).

- log.csv
  - Total rows: 1,758.
  - Distinct cmdb_id (sample) includes pods such as adservice-0/1/2, cartservice-*, checkoutservice-*, frontend-*, paymentservice-*, productcatalogservice-*, recommendationservice-*, shippingservice-*, and redis-cart-0.
  - Distinct log names: log.error_count and log.row_count.
  - Timestamp range (UTC+8): 2022-03-21T01:30:00+08:00 through 2022-03-21T02:00:00+08:00.
  - Top rows show per-pod log row_count and error_count at timestamp 1647797400 (error_count mostly 0).

- error_logs.csv
  - Total rows: 0 — no error log records were present.
  - Distinct cmdb_id: none.
  - No timestamps or earliest rows to report.

Overall conclusion: metric, trace, and log data are present and cover 01:00–02:00 (metrics) and 01:30–02:00 (trace/log) windows; error logs are empty. If you want deeper analysis (anomalies, top noisy components, or time-aligned joins across datasets), tell me which analysis to run next.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'row_count': 321463, 'distinct_cmdb_id': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1'], 'distinct_kpi_name': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.MarkSweepCompact'], 'timestamp_min_UTC+8': '2022-03-21T01:00:00+08:00', 'timestamp_max_UTC+8': '2022-03-21T02:00:00+08:00', 'top_10_rows': [{'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.MarkSweepCompact', 'value': 539245384.0}, {'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_id.Copy', 'value': 37350.083333333336}, {'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_id.MarkSweepCompact', 'value': 8.0}, {'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.Copy', 'value': 23330816.0}, {'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.MarkSweepCompact', 'value': 23330816.0}, {'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.Copy', 'value': 3932160.0}, {'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.MarkSweepCompact', 'value': 3932160.0}, {'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.Copy', 'value': 27656192.0}, {'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_MemoryPool_Usage_committed.Compressed_Class_Space', 'value': 3932160.0}, {'timestamp': 1647795600, 'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_MemoryPool_Usage_committed.Eden_Space', 'value': 27656192.0}]}, 'trace.csv': {'row_count': 29328, 'distinct_cmdb_id': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0', 'checkoutservice2-0', 'emailservice-2', 'emailservice2-0', 'paymentservice2-0'], 'distinct_trace_name': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.error_rate', 'trace.self.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.to_cartservice-0.duration_mean', 'trace.to_cartservice-0.duration_p95', 'trace.to_cartservice-0.error_rate', 'trace.to_cartservice-0.row_count', 'trace.to_cartservice-1.duration_mean', 'trace.to_cartservice-1.duration_p95', 'trace.to_cartservice-1.error_rate', 'trace.to_cartservice-1.row_count', 'trace.to_cartservice-2.duration_mean', 'trace.to_cartservice-2.duration_p95', 'trace.to_cartservice-2.error_rate', 'trace.to_cartservice-2.row_count', 'trace.to_currencyservice-0.duration_mean', 'trace.to_currencyservice-0.duration_p95'], 'timestamp_min_UTC+8': '2022-03-21T01:30:00+08:00', 'timestamp_max_UTC+8': '2022-03-21T01:59:00+08:00', 'top_10_rows': [{'timestamp': 1647797400, 'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_checkoutservice-0.duration_mean', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_checkoutservice-0.duration_p95', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_checkoutservice-0.error_rate', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_checkoutservice-0.row_count', 'value': 1.0}, {'timestamp': 1647797400, 'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_checkoutservice-1.duration_mean', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_checkoutservice-1.duration_p95', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_checkoutservice-1.error_rate', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'cartservice-0', 'trace_name': 'trace.from_checkoutservice-1.row_count', 'value': 1.0}, {'timestamp': 1647797400, 'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-2.duration_mean', 'value': 1.5e-05}, {'timestamp': 1647797400, 'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-2.duration_p95', 'value': 2e-05}]}, 'log.csv': {'row_count': 1758, 'distinct_cmdb_id': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'emailservice-2', 'redis-cart-0'], 'distinct_log_name': ['log.error_count', 'log.row_count'], 'timestamp_min_UTC+8': '2022-03-21T01:30:00+08:00', 'timestamp_max_UTC+8': '2022-03-21T02:00:00+08:00', 'top_10_rows': [{'timestamp': 1647797400, 'cmdb_id': 'adservice-0', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'adservice-0', 'log_name': 'log.row_count', 'value': 48.0}, {'timestamp': 1647797400, 'cmdb_id': 'cartservice-2', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'cartservice-2', 'log_name': 'log.row_count', 'value': 327.0}, {'timestamp': 1647797400, 'cmdb_id': 'checkoutservice-0', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'checkoutservice-0', 'log_name': 'log.row_count', 'value': 50.0}, {'timestamp': 1647797400, 'cmdb_id': 'checkoutservice-1', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'checkoutservice-1', 'log_name': 'log.row_count', 'value': 17.0}, {'timestamp': 1647797400, 'cmdb_id': 'checkoutservice-2', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647797400, 'cmdb_id': 'checkoutservice-2', 'log_name': 'log.row_count', 'value': 34.0}]}, 'error_logs.csv': {'row_count': 0, 'distinct_cmdb_id': [], 'timestamp_min_UTC+8': 'NaT', 'timestamp_max_UTC+8': 'NaT', 'earliest_10_rows': []}}```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure kpi_name and cmdb_id are string-typed for consistent processing
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)
df_metric['cmdb_id'] = df_metric['cmdb_id'].astype(str)

# Aggregate: row counts per kpi_name and sample up to 5 distinct cmdb_ids
gb = df_metric.groupby('kpi_name')['cmdb_id'].agg(lambda s: sorted(pd.Series(s.dropna().unique()))).reset_index()
gb['row_count'] = gb['kpi_name'].map(df_metric['kpi_name'].value_counts())

# Sort kpi names alphabetically and limit to first 200
gb_sorted = gb.sort_values('kpi_name').reset_index(drop=True)
gb_limited = gb_sorted.head(200).copy()

# Build JSON-serializable records
kpi_records = []
for _, row in gb_limited.iterrows():
    sample_cmdbs = row['cmdb_id'][:5] if isinstance(row['cmdb_id'], (list, tuple, pd.Series)) else [row['cmdb_id']]
    kpi_records.append({
        'kpi_name': row['kpi_name'],
        'row_count': int(row['row_count']),
        'sample_cmdb_ids': list(sample_cmdbs)
    })

result = {
    'total_kpi_names': int(df_metric['kpi_name'].nunique()),
    'kpis': kpi_records
}

result

```
Out[2]:
```
summary = (
    "Metric KPI overview (plain English):\n\n"
    "- Total distinct kpi_name values: 1,384. The output returned the first 200 names sorted alphabetically.\n"
    "- Key KPI categories observed among the first 200 names:\n"
    "  * Application traffic: app.grpc.* (549 rows) and app.http.* (122 rows) — service-level RPC/HTTP traffic/latency metrics.\n"
    "  * Container/node resource metrics: many container.node-5.* (typically 366 rows each) and container.node-6.* (often 2,074+ rows) KPIs covering CPU, memory, network, filesystem and I/O.\n"
    "  * Mesh / Istio traffic metrics: mesh.destination.* (various istio_requests, istio_response_bytes, durations) with groups of 61, 122, or 183 rows depending on the metric.\n"
    "  * JVM/runtime metrics (runtime.java_lang_*, GarbageCollector_*, MemoryPool_*) are present elsewhere in metric.csv (seen in earlier top rows).\n"
    "- Sample components where these KPIs appear include both services and pods (e.g., adservice, cartservice, checkoutservice, frontend, various pod suffixes like -0/-1/-2, and node-1..node-6).\n\n"
    "Conclusion / suggestion: for resource or traffic-related investigations, focus on container.node-* KPIs (CPU/memory/network/fs) and mesh.destination/istio KPIs (requests/bytes/durations), plus app.grpc/app.http KPIs for service-level traffic and latency."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'total_kpi_names': 1384, 'kpis': [{'kpi_name': 'app.grpc.count', 'row_count': 549, 'sample_cmdb_ids': ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice']}, {'kpi_name': 'app.grpc.mrt', 'row_count': 549, 'sample_cmdb_ids': ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice']}, {'kpi_name': 'app.grpc.rr', 'row_count': 549, 'sample_cmdb_ids': ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice']}, {'kpi_name': 'app.grpc.sr', 'row_count': 549, 'sample_cmdb_ids': ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice']}, {'kpi_name': 'app.http.count', 'row_count': 122, 'sample_cmdb_ids': ['adservice', 'frontend']}, {'kpi_name': 'app.http.mrt', 'row_count': 122, 'sample_cmdb_ids': ['adservice', 'frontend']}, {'kpi_name': 'app.http.rr', 'row_count': 122, 'sample_cmdb_ids': ['adservice', 'frontend']}, {'kpi_name': 'app.http.sr', 'row_count': 122, 'sample_cmdb_ids': ['adservice', 'frontend']}, {'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_file_descriptors', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_reads_MB./dev/vda', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_sector_writes./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_usage_MB./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_write_seconds./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_writes./dev/vda', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_writes./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_writes_MB./dev/vda', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_writes_merged./dev/vda1', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_last_seen', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_cache', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failcnt', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failures.container.pgfault', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failures.container.pgmajfault', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failures.hierarchy.pgfault', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_mapped_file', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_max_usage_MB', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_rss', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_swap', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_usage_MB', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_working_set_MB', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_receive_MB.eth0', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_receive_errors.eth0', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_receive_packets.eth0', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_receive_packets_dropped.eth0', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_transmit_MB.eth0', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_transmit_errors.eth0', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_transmit_packets.eth0', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_transmit_packets_dropped.eth0', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_sockets', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_cpu_period', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_cpu_quota', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_cpu_shares', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_memory_limit_MB', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_memory_reservation_limit_MB', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_memory_swap_limit_MB', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_start_time_seconds', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.iowaiting', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.running', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.sleeping', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.stopped', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.uninterruptible', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_threads', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_threads_max', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_ulimits_soft.max_open_files', 'row_count': 366, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_file_descriptors', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_io_current./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_io_time_seconds./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_io_time_weighted_seconds./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_limit_MB./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_read_seconds./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_reads./dev/vda', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_reads./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_reads_MB./dev/vda', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_reads_merged./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_sector_reads./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_sector_writes./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_usage_MB./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_write_seconds./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_writes./dev/vda', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_writes./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_writes_MB./dev/vda', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_writes_merged./dev/vda1', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_last_seen', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_cache', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failcnt', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failures.container.pgfault', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failures.container.pgmajfault', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failures.hierarchy.pgfault', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failures.hierarchy.pgmajfault', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_mapped_file', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_max_usage_MB', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_rss', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_swap', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_working_set_MB', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_receive_MB.eth0', 'row_count': 2196, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_receive_errors.eth0', 'row_count': 2196, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'row_count': 2196, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_receive_packets_dropped.eth0', 'row_count': 2196, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_transmit_MB.eth0', 'row_count': 2196, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_transmit_errors.eth0', 'row_count': 2196, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'row_count': 2196, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_transmit_packets_dropped.eth0', 'row_count': 2196, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_sockets', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_cpu_period', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_cpu_quota', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_cpu_shares', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_memory_limit_MB', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_memory_reservation_limit_MB', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_memory_swap_limit_MB', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_start_time_seconds', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.iowaiting', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.running', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.sleeping', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.stopped', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.uninterruptible', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_threads', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_threads_max', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_ulimits_soft.max_open_files', 'row_count': 2074, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'mesh.destination.cartservice.redis-cart.istio_tcp_connections_closed.-', 'row_count': 61, 'sample_cmdb_ids': ['redis-cart-0']}, {'kpi_name': 'mesh.destination.cartservice.redis-cart.istio_tcp_connections_opened.-', 'row_count': 61, 'sample_cmdb_ids': ['redis-cart-0']}, {'kpi_name': 'mesh.destination.cartservice.redis-cart.istio_tcp_received_bytes.-', 'row_count': 61, 'sample_cmdb_ids': ['redis-cart-0']}, {'kpi_name': 'mesh.destination.cartservice.redis-cart.istio_tcp_sent_bytes.-', 'row_count': 61, 'sample_cmdb_ids': ['redis-cart-0']}, {'kpi_name': 'mesh.destination.cartservice2.redis-cart2.istio_tcp_connections_closed.-', 'row_count': 61, 'sample_cmdb_ids': ['redis-cart2-0']}, {'kpi_name': 'mesh.destination.cartservice2.redis-cart2.istio_tcp_connections_opened.-', 'row_count': 61, 'sample_cmdb_ids': ['redis-cart2-0']}, {'kpi_name': 'mesh.destination.cartservice2.redis-cart2.istio_tcp_received_bytes.-', 'row_count': 61, 'sample_cmdb_ids': ['redis-cart2-0']}, {'kpi_name': 'mesh.destination.cartservice2.redis-cart2.istio_tcp_sent_bytes.-', 'row_count': 61, 'sample_cmdb_ids': ['redis-cart2-0']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_request_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_request_bytes.grpc.200.14.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_request_bytes.grpc.200.9.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_request_duration_milliseconds.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_request_duration_milliseconds.grpc.200.14.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_request_duration_milliseconds.grpc.200.9.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_request_messages', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_requests.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_requests.grpc.200.14.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_requests.grpc.200.9.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_response_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_response_bytes.grpc.200.14.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_response_bytes.grpc.200.9.0', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.cartservice.istio_response_messages', 'row_count': 183, 'sample_cmdb_ids': ['cartservice-0', 'cartservice-1', 'cartservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_request_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['currencyservice-0', 'currencyservice-1', 'currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_request_bytes.grpc.200.14.0', 'row_count': 61, 'sample_cmdb_ids': ['currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_request_duration_milliseconds.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['currencyservice-0', 'currencyservice-1', 'currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_request_duration_milliseconds.grpc.200.14.0', 'row_count': 61, 'sample_cmdb_ids': ['currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_request_messages', 'row_count': 183, 'sample_cmdb_ids': ['currencyservice-0', 'currencyservice-1', 'currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_requests.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['currencyservice-0', 'currencyservice-1', 'currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_requests.grpc.200.14.0', 'row_count': 61, 'sample_cmdb_ids': ['currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_response_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['currencyservice-0', 'currencyservice-1', 'currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_response_bytes.grpc.200.14.0', 'row_count': 61, 'sample_cmdb_ids': ['currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.currencyservice.istio_response_messages', 'row_count': 183, 'sample_cmdb_ids': ['currencyservice-0', 'currencyservice-1', 'currencyservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_request_bytes.grpc.0.2.0', 'row_count': 122, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_request_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-1', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_request_duration_milliseconds.grpc.0.2.0', 'row_count': 122, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_request_duration_milliseconds.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-1', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_request_messages', 'row_count': 183, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-1', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_requests.grpc.0.2.0', 'row_count': 122, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_requests.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-1', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_response_bytes.grpc.0.2.0', 'row_count': 122, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_response_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-1', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.emailservice.istio_response_messages', 'row_count': 183, 'sample_cmdb_ids': ['emailservice-0', 'emailservice-1', 'emailservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_request_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['paymentservice-0', 'paymentservice-1', 'paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_request_bytes.grpc.200.14.0', 'row_count': 61, 'sample_cmdb_ids': ['paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_request_duration_milliseconds.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['paymentservice-0', 'paymentservice-1', 'paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_request_duration_milliseconds.grpc.200.14.0', 'row_count': 61, 'sample_cmdb_ids': ['paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_request_messages', 'row_count': 183, 'sample_cmdb_ids': ['paymentservice-0', 'paymentservice-1', 'paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_requests.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['paymentservice-0', 'paymentservice-1', 'paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_requests.grpc.200.14.0', 'row_count': 61, 'sample_cmdb_ids': ['paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_response_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['paymentservice-0', 'paymentservice-1', 'paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_response_bytes.grpc.200.14.0', 'row_count': 61, 'sample_cmdb_ids': ['paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.paymentservice.istio_response_messages', 'row_count': 183, 'sample_cmdb_ids': ['paymentservice-0', 'paymentservice-1', 'paymentservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_request_bytes.grpc.0.2.0', 'row_count': 122, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_request_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_request_bytes.grpc.200.2.0', 'row_count': 183, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_request_duration_milliseconds.grpc.0.2.0', 'row_count': 122, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_request_duration_milliseconds.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_request_duration_milliseconds.grpc.200.2.0', 'row_count': 183, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_request_messages', 'row_count': 183, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_requests.grpc.0.2.0', 'row_count': 122, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_requests.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_requests.grpc.200.2.0', 'row_count': 183, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_response_bytes.grpc.0.2.0', 'row_count': 122, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1']}, {'kpi_name': 'mesh.destination.checkoutservice.productcatalogservice.istio_response_bytes.grpc.200.0.0', 'row_count': 183, 'sample_cmdb_ids': ['productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2']}]}```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# reuse df_metric from prior state if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)
df_metric['cmdb_id'] = df_metric['cmdb_id'].astype(str)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Filter rows where kpi_name contains 'container.node-'
mask = df_metric['kpi_name'].str.contains('container.node-', na=False)
df_cn = df_metric[mask].copy()

# Group and compute aggregates using the full series per kpi_name
grouped = df_cn.groupby('kpi_name')

agg_list = []
for kpi, grp in grouped:
    total_points = int(len(grp))
    vals = grp['value'].dropna()
    global_p50 = float(vals.quantile(0.50)) if not vals.empty else float('nan')
    global_p95 = float(vals.quantile(0.95)) if not vals.empty else float('nan')
    sample_cmdb_ids = sorted(grp['cmdb_id'].dropna().unique())[:5]
    agg_list.append({
        'kpi_name': kpi,
        'total_points': total_points,
        'global_p50': global_p50,
        'global_p95': global_p95,
        'sample_cmdb_ids': sample_cmdb_ids
    })

# Sort alphabetically by kpi_name and limit to first 200
agg_sorted = sorted(agg_list, key=lambda x: x['kpi_name'])
agg_limited = agg_sorted[:200]

result = {
    'total_kpis': len(agg_limited),
    'kpi_thresholds': agg_limited
}

result

```
Out[3]:
```
Summary (plain English):

- Scope: 128 distinct kpi_name values contain "container.node-". These are per-container/node resource metrics (CPU, memory, filesystem, network, sockets, tasks, ulimits, etc.).

- Data density:
  - Many node-5 KPIs have 366 data points each (sample components: adservice-2, cartservice2-0, checkoutservice-2, frontend-1, frontend-2).
  - Many node-6 KPIs have ~2,074 (or 2,196 for some network metrics) data points each (sample components: adservice-0, adservice-1, adservice2-0, cartservice-0, cartservice-1).
  - The differing counts indicate node-6 metrics are more abundant in the dataset than node-5.

- Typical central and tail values (examples):
  - CPU usage (container_cpu_usage_seconds):
    - node-5: p50 ≈ 0.066, p95 ≈ 0.289
    - node-6: p50 ≈ 0.095, p95 ≈ 3.16 (node-6 tail is much higher)
  - Memory usage (container_memory_usage_MB):
    - node-5: p50 ≈ 42.06 MB, p95 ≈ 159.63 MB
    - node-6: p50 ≈ 78.68 MB, p95 ≈ 255.94 MB
  - Memory RSS (container_memory_rss) p95 can be very large (e.g., node-6 p95 ≈ 224,296,960).
  - Network packets (container_network_receive_packets.eth0):
    - node-5 p95 ≈ 1,106 packets
    - node-6 p95 ≈ 2,576 packets
  - Filesystem limit (container_fs_limit_MB./dev/vda1) is effectively constant (p50 = p95 ≈ 604,630.74).
  - Many I/O and task-state metrics have p50 = p95 = 0, indicating sparse/zero activity for those KPIs.

- Where these KPIs appear (sample cmdb_ids): adservice-0/1/2, adservice2-0, cartservice-0/1/2, cartservice2-0, checkoutservice-2, frontend-1/2, etc.

- Actionable note: For resource-related troubleshooting focus on KPIs with large p95 tails — e.g., container_cpu_usage_seconds, container_memory_usage_MB / memory_rss, and network packet counts — especially on node-6 where tails are larger. These indicate which containers/nodes experience occasional high CPU, memory, or network load.

The original code execution output of IPython Kernel is also provided below for reference:

{'total_kpis': 128, 'kpi_thresholds': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'total_points': 366, 'global_p50': 58.83333333337214, 'global_p95': 179.62499999999818, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 1.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.067962528332373, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'total_points': 366, 'global_p50': 0.0200000000000244, 'global_p95': 0.0900000000001455, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'total_points': 366, 'global_p50': 0.0658940029166501, 'global_p95': 0.28878242800010406, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'total_points': 366, 'global_p50': 0.0349999999999681, 'global_p95': 0.1500000000000909, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_file_descriptors', 'total_points': 366, 'global_p50': 17.0, 'global_p95': 162.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'total_points': 366, 'global_p50': 604630.73828125, 'global_p95': 604630.73828125, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_reads_MB./dev/vda', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_sector_writes./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_usage_MB./dev/vda1', 'total_points': 366, 'global_p50': 40.0556640625, 'global_p95': 56.982421875, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_write_seconds./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_writes./dev/vda', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_writes./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_writes_MB./dev/vda', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_fs_writes_merged./dev/vda1', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_last_seen', 'total_points': 366, 'global_p50': 1647797418.0, 'global_p95': 1647799038.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_cache', 'total_points': 366, 'global_p50': 8069120.0, 'global_p95': 43753472.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failcnt', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failures.container.pgfault', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 33.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failures.container.pgmajfault', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failures.hierarchy.pgfault', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 33.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_mapped_file', 'total_points': 366, 'global_p50': 5812224.0, 'global_p95': 39333888.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_max_usage_MB', 'total_points': 366, 'global_p50': 192.0, 'global_p95': 600.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_rss', 'total_points': 366, 'global_p50': 17885184.0, 'global_p95': 148389888.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_swap', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_usage_MB', 'total_points': 366, 'global_p50': 42.0625, 'global_p95': 159.6328125, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_memory_working_set_MB', 'total_points': 366, 'global_p50': 31.20703125, 'global_p95': 156.65625, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_receive_MB.eth0', 'total_points': 366, 'global_p50': 0.039689540863037095, 'global_p95': 0.228684544563265, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_receive_errors.eth0', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_receive_packets.eth0', 'total_points': 366, 'global_p50': 354.0833333332557, 'global_p95': 1106.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_receive_packets_dropped.eth0', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_transmit_MB.eth0', 'total_points': 366, 'global_p50': 1.4496791362762451, 'global_p95': 4.453991651535034, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_transmit_errors.eth0', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_transmit_packets.eth0', 'total_points': 366, 'global_p50': 302.41666666651145, 'global_p95': 1359.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_network_transmit_packets_dropped.eth0', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_sockets', 'total_points': 366, 'global_p50': 7.0, 'global_p95': 12.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_cpu_period', 'total_points': 366, 'global_p50': 100000.0, 'global_p95': 100000.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_cpu_quota', 'total_points': 366, 'global_p50': 40000.0, 'global_p95': 60000.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_cpu_shares', 'total_points': 366, 'global_p50': 204.0, 'global_p95': 409.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_memory_limit_MB', 'total_points': 366, 'global_p50': 256.0, 'global_p95': 600.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_memory_reservation_limit_MB', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_spec_memory_swap_limit_MB', 'total_points': 366, 'global_p50': 256.0, 'global_p95': 600.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_start_time_seconds', 'total_points': 366, 'global_p50': 1647144389.5, 'global_p95': 1647781586.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.iowaiting', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.running', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.sleeping', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.stopped', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_tasks_state.uninterruptible', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_threads', 'total_points': 366, 'global_p50': 27.0, 'global_p95': 29.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_threads_max', 'total_points': 366, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-5.container_ulimits_soft.max_open_files', 'total_points': 366, 'global_p50': 65536.0, 'global_p95': 65536.0, 'sample_cmdb_ids': ['adservice-2', 'cartservice2-0', 'checkoutservice-2', 'frontend-1', 'frontend-2']}, {'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'total_points': 2074, 'global_p50': 93.5, 'global_p95': 591.8499999999999, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 1.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.06268879764966169, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'total_points': 2074, 'global_p50': 0.0150000000000432, 'global_p95': 3.0417499999999222, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'total_points': 2074, 'global_p50': 0.0954640740001195, 'global_p95': 3.1599951171000837, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'total_points': 2074, 'global_p50': 0.0700000000001637, 'global_p95': 0.2299999999999897, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_file_descriptors', 'total_points': 2074, 'global_p50': 11.0, 'global_p95': 163.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_io_current./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_io_time_seconds./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_io_time_weighted_seconds./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_limit_MB./dev/vda1', 'total_points': 2074, 'global_p50': 604630.73828125, 'global_p95': 604630.73828125, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_read_seconds./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_reads./dev/vda', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 14920.974999999999, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_reads./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_reads_MB./dev/vda', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 4669.03125, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_reads_merged./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_sector_reads./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_sector_writes./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_usage_MB./dev/vda1', 'total_points': 2074, 'global_p50': 42.234375, 'global_p95': 655.658984375, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_write_seconds./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_writes./dev/vda', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_writes./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_writes_MB./dev/vda', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_fs_writes_merged./dev/vda1', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_last_seen', 'total_points': 2074, 'global_p50': 1647797440.0, 'global_p95': 1647799060.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_cache', 'total_points': 2074, 'global_p50': 25374720.0, 'global_p95': 194876825.6, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failcnt', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 10.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failures.container.pgfault', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 15868.049999999997, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failures.container.pgmajfault', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failures.hierarchy.pgfault', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 15868.049999999997, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_failures.hierarchy.pgmajfault', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_mapped_file', 'total_points': 2074, 'global_p50': 1689600.0, 'global_p95': 14057472.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_max_usage_MB', 'total_points': 2074, 'global_p50': 256.00390625, 'global_p95': 1025.00390625, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_rss', 'total_points': 2074, 'global_p50': 37273600.0, 'global_p95': 224296960.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_swap', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'total_points': 2074, 'global_p50': 78.68359375, 'global_p95': 255.9427734375, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_memory_working_set_MB', 'total_points': 2074, 'global_p50': 63.8955078125, 'global_p95': 233.703125, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_receive_MB.eth0', 'total_points': 2196, 'global_p50': 0.045400857925415, 'global_p95': 0.4381629228591919, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_receive_errors.eth0', 'total_points': 2196, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'total_points': 2196, 'global_p50': 447.0, 'global_p95': 2576.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_receive_packets_dropped.eth0', 'total_points': 2196, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_transmit_MB.eth0', 'total_points': 2196, 'global_p50': 0.81846022605896, 'global_p95': 4.20786714553833, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_transmit_errors.eth0', 'total_points': 2196, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'total_points': 2196, 'global_p50': 299.0, 'global_p95': 2504.375, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_network_transmit_packets_dropped.eth0', 'total_points': 2196, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_sockets', 'total_points': 2074, 'global_p50': 4.0, 'global_p95': 12.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_cpu_period', 'total_points': 2074, 'global_p50': 100000.0, 'global_p95': 100000.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_cpu_quota', 'total_points': 2074, 'global_p50': 40000.0, 'global_p95': 60000.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_cpu_shares', 'total_points': 2074, 'global_p50': 204.0, 'global_p95': 409.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_memory_limit_MB', 'total_points': 2074, 'global_p50': 256.0, 'global_p95': 1025.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_memory_reservation_limit_MB', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_spec_memory_swap_limit_MB', 'total_points': 2074, 'global_p50': 256.0, 'global_p95': 1025.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_start_time_seconds', 'total_points': 2074, 'global_p50': 1647144380.0, 'global_p95': 1647758894.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.iowaiting', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.running', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.sleeping', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.stopped', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_tasks_state.uninterruptible', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_threads', 'total_points': 2074, 'global_p50': 30.0, 'global_p95': 37.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_threads_max', 'total_points': 2074, 'global_p50': 0.0, 'global_p95': 0.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}, {'kpi_name': 'container.node-6.container_ulimits_soft.max_open_files', 'total_points': 2074, 'global_p50': 1048576.0, 'global_p95': 1048576.0, 'sample_cmdb_ids': ['adservice-0', 'adservice-1', 'adservice2-0', 'cartservice-0', 'cartservice-1']}]}```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric from previous steps
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure correct types
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)
df_metric['cmdb_id'] = df_metric['cmdb_id'].astype(str)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')

tz = pytz.timezone('Asia/Shanghai')

# Patterns to examine (exact substrings to match within kpi_name)
patterns = [
    'container.node-6.container_cpu_usage_seconds',
    'container.node-6.container_memory_usage_MB',
    'container.node-6.container_network_receive_packets.eth0',
    'container.node-6.container_network_transmit_packets.eth0'
]

# Identify kpi_names that contain any of the patterns
all_kpis = sorted(df_metric['kpi_name'].dropna().unique())
selected_kpis = [k for k in all_kpis if any(p in k for p in patterns)]
# Compute global p95 for each selected kpi_name using full series (no time filtering)
global_p95_map = {}
for k in selected_kpis:
    vals = df_metric.loc[df_metric['kpi_name'] == k, 'value'].dropna()
    global_p95_map[k] = float(vals.quantile(0.95)) if not vals.empty else 0.0

# Filter rows for selected kpis AND timestamp window (inclusive)
start_ts = 1647797400
end_ts = 1647799200
mask_sel = df_metric['kpi_name'].isin(selected_kpis) & (df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)
df_window = df_metric.loc[mask_sel].copy()

# Group by (kpi_name, cmdb_id) and compute required aggregates
records = []
grouped = df_window.groupby(['kpi_name', 'cmdb_id'], sort=True)
for (kpi, cmdb), grp in grouped:
    points_in_window = int(len(grp))
    gp95 = global_p95_map.get(kpi, 0.0)
    # anomalies: value >= global_p95 (use >= as requested)
    anoms = grp.loc[grp['value'] >= gp95]
    anomalies_count = int(len(anoms))
    if anomalies_count > 0:
        earliest_anom_ts = int(anoms['timestamp'].min())
        earliest_anom_iso = pd.to_datetime(earliest_anom_ts, unit='s', utc=True).tz_convert(tz).isoformat()
    else:
        earliest_anom_iso = None
    # max value in window
    max_value_in_window = float(grp['value'].max()) if not grp['value'].dropna().empty else None
    # severity_ratio
    if gp95 == 0 or max_value_in_window is None:
        severity_ratio = None
    else:
        severity_ratio = float(max_value_in_window / gp95)
    records.append({
        'kpi_name': kpi,
        'cmdb_id': cmdb,
        'points_in_window': points_in_window,
        'anomalies_count': anomalies_count,
        'earliest_anomaly_timestamp': earliest_anom_iso,
        'max_value_in_window': max_value_in_window,
        'global_p95': gp95,
        'severity_ratio': severity_ratio
    })

# Summary counts
groups_examined = len(records)
groups_with_anomalies = sum(1 for r in records if r['anomalies_count'] > 0)

# Sort top anomalies by severity_ratio descending (treat None as -1 for sorting), limit to 20
for r in records:
    r['_severity_sort'] = r['severity_ratio'] if r['severity_ratio'] is not None else -1.0

top_anomalies = sorted(records, key=lambda x: x['_severity_sort'], reverse=True)[:20]
# Remove helper sort key before returning
for r in top_anomalies:
    r.pop('_severity_sort', None)

result = {
    'summary': {
        'groups_examined': groups_examined,
        'groups_with_anomalies': groups_with_anomalies
    },
    'top_anomalies': top_anomalies
}

result

```
Out[5]:
```
Summary of anomaly detection (plain English):

- Scope: I examined 140 (kpi_name, cmdb_id) groups for four node-6 KPIs within the time window 2022-03-21 01:30:00–02:00:00 (UTC+8). 13 groups had values >= their global p95.

- Highest-severity findings (top examples):
  1. container.node-6.container_network_transmit_packets.eth0 — productcatalogservice2-0
     - points_in_window: 31, anomalies: 28, earliest anomaly: 2022-03-21T01:31:00+08:00
     - max value: 3848.0, global p95: 2504.375, severity ≈ 1.54×
  2. container.node-6.container_network_receive_packets.eth0 — productcatalogservice2-0
     - points_in_window: 31, anomalies: 26, earliest anomaly: 2022-03-21T01:31:00+08:00
     - max value: 3729.5, global p95: 2576.0, severity ≈ 1.45×
  3. container.node-6.container_cpu_usage_seconds — currencyservice-0
     - points_in_window: 31, anomalies: 26, earliest anomaly: 2022-03-21T01:30:00+08:00
     - max value: 4.239, global p95: 3.160, severity ≈ 1.34×
  4. container.node-6.container_cpu_usage_seconds — currencyservice-2
     - points_in_window: 31, anomalies: 10, earliest anomaly: 2022-03-21T01:30:00+08:00
     - severity ≈ 1.17×
  5. Several productcatalogservice pods (productcatalogservice-0/1/2 and productcatalogservice2-0) show elevated receive/transmit packet counts (severity ≈ 1.13–1.17×).
  6. container.node-6.container_memory_usage_MB — currencyservice-0 and currencyservice-2
     - both have small exceedances just above global p95 (severity ≈ 1.0002×), with many anomaly points (24–28/31).

- Counts: groups_examined = 140; groups_with_anomalies = 13.

- Interpretation and recommended next steps:
  - Network spikes: productcatalogservice2-0 (and other productcatalogservice pods) had substantial network packet spikes (transmit & receive) during the window. Investigate service traffic patterns, recent deployments, pod-to-pod traffic, or client load for productcatalogservice.
  - CPU & memory on currencyservice: currencyservice-0 (and -2) show both CPU and slight memory exceedances — check application logs, request rates, and recent scaling or GC activity for currencyservice pods.
  - These anomalies are sustained across many minutes (many anomaly points out of 31), so they are not single outliers.
  - Next actions: correlate with trace/log data for these pods in the same window, check network metrics at node-level, and review recent changes/deploys for productcatalogservice and currencyservice.

If you want, I can now fetch the per-pod trace/log rows for these top components during the same window to help root-cause investigation.

The original code execution output of IPython Kernel is also provided below for reference:

{'summary': {'groups_examined': 140, 'groups_with_anomalies': 13}, 'top_anomalies': [{'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'cmdb_id': 'productcatalogservice2-0', 'points_in_window': 31, 'anomalies_count': 28, 'earliest_anomaly_timestamp': '2022-03-21T01:31:00+08:00', 'max_value_in_window': 3848.0, 'global_p95': 2504.375, 'severity_ratio': 1.5365111055652607}, {'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'cmdb_id': 'productcatalogservice2-0', 'points_in_window': 31, 'anomalies_count': 26, 'earliest_anomaly_timestamp': '2022-03-21T01:31:00+08:00', 'max_value_in_window': 3729.5, 'global_p95': 2576.0, 'severity_ratio': 1.4477872670807452}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'cmdb_id': 'currencyservice-0', 'points_in_window': 31, 'anomalies_count': 26, 'earliest_anomaly_timestamp': '2022-03-21T01:30:00+08:00', 'max_value_in_window': 4.239449702999991, 'global_p95': 3.1599951171000837, 'severity_ratio': 1.3416000803477566}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'cmdb_id': 'currencyservice-2', 'points_in_window': 31, 'anomalies_count': 10, 'earliest_anomaly_timestamp': '2022-03-21T01:30:00+08:00', 'max_value_in_window': 3.6991392260001694, 'global_p95': 3.1599951171000837, 'severity_ratio': 1.170615487974189}, {'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'cmdb_id': 'productcatalogservice-0', 'points_in_window': 31, 'anomalies_count': 12, 'earliest_anomaly_timestamp': '2022-03-21T01:31:00+08:00', 'max_value_in_window': 3009.5, 'global_p95': 2576.0, 'severity_ratio': 1.1682841614906831}, {'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'cmdb_id': 'productcatalogservice-1', 'points_in_window': 31, 'anomalies_count': 12, 'earliest_anomaly_timestamp': '2022-03-21T01:31:00+08:00', 'max_value_in_window': 2922.5, 'global_p95': 2504.375, 'severity_ratio': 1.1669578238083353}, {'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'cmdb_id': 'productcatalogservice-1', 'points_in_window': 31, 'anomalies_count': 12, 'earliest_anomaly_timestamp': '2022-03-21T01:31:00+08:00', 'max_value_in_window': 2992.5, 'global_p95': 2576.0, 'severity_ratio': 1.1616847826086956}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'cmdb_id': 'currencyservice2-0', 'points_in_window': 31, 'anomalies_count': 18, 'earliest_anomaly_timestamp': '2022-03-21T01:30:00+08:00', 'max_value_in_window': 3.667496106500039, 'global_p95': 3.1599951171000837, 'severity_ratio': 1.1606018270894314}, {'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'cmdb_id': 'productcatalogservice-0', 'points_in_window': 31, 'anomalies_count': 11, 'earliest_anomaly_timestamp': '2022-03-21T01:31:00+08:00', 'max_value_in_window': 2884.5, 'global_p95': 2504.375, 'severity_ratio': 1.1517843773396557}, {'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'cmdb_id': 'productcatalogservice-2', 'points_in_window': 31, 'anomalies_count': 14, 'earliest_anomaly_timestamp': '2022-03-21T01:32:00+08:00', 'max_value_in_window': 2832.0, 'global_p95': 2504.375, 'severity_ratio': 1.130821063139506}, {'kpi_name': 'container.node-6.container_network_receive_packets.eth0', 'cmdb_id': 'productcatalogservice-2', 'points_in_window': 31, 'anomalies_count': 14, 'earliest_anomaly_timestamp': '2022-03-21T01:32:00+08:00', 'max_value_in_window': 2898.5, 'global_p95': 2576.0, 'severity_ratio': 1.125194099378882}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'cmdb_id': 'currencyservice-0', 'points_in_window': 31, 'anomalies_count': 28, 'earliest_anomaly_timestamp': '2022-03-21T01:30:00+08:00', 'max_value_in_window': 256.0, 'global_p95': 255.9427734375, 'severity_ratio': 1.0002235912416335}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'cmdb_id': 'currencyservice-2', 'points_in_window': 31, 'anomalies_count': 24, 'earliest_anomaly_timestamp': '2022-03-21T01:32:00+08:00', 'max_value_in_window': 255.998046875, 'global_p95': 255.9427734375, 'severity_ratio': 1.0002159601412364}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'cmdb_id': 'adservice-0', 'points_in_window': 31, 'anomalies_count': 0, 'earliest_anomaly_timestamp': None, 'max_value_in_window': 250.15625, 'global_p95': 255.9427734375, 'severity_ratio': 0.9773913388536714}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'cmdb_id': 'adservice2-0', 'points_in_window': 31, 'anomalies_count': 0, 'earliest_anomaly_timestamp': None, 'max_value_in_window': 240.34375, 'global_p95': 255.9427734375, 'severity_ratio': 0.9390526904589115}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'cmdb_id': 'cartservice-0', 'points_in_window': 31, 'anomalies_count': 0, 'earliest_anomaly_timestamp': None, 'max_value_in_window': 164.56640625, 'global_p95': 255.9427734375, 'severity_ratio': 0.6429812572543149}, {'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'cmdb_id': 'frontend-0', 'points_in_window': 31, 'anomalies_count': 0, 'earliest_anomaly_timestamp': None, 'max_value_in_window': 1584.0, 'global_p95': 2504.375, 'severity_ratio': 0.6324931370102321}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'cmdb_id': 'adservice-1', 'points_in_window': 31, 'anomalies_count': 0, 'earliest_anomaly_timestamp': None, 'max_value_in_window': 158.853515625, 'global_p95': 255.9427734375, 'severity_ratio': 0.6206602885929549}, {'kpi_name': 'container.node-6.container_network_transmit_packets.eth0', 'cmdb_id': 'frontend2-0', 'points_in_window': 31, 'anomalies_count': 0, 'earliest_anomaly_timestamp': None, 'max_value_in_window': 1463.5, 'global_p95': 2504.375, 'severity_ratio': 0.5843773396556027}, {'kpi_name': 'container.node-6.container_memory_usage_MB', 'cmdb_id': 'cartservice-1', 'points_in_window': 31, 'anomalies_count': 0, 'earliest_anomaly_timestamp': None, 'max_value_in_window': 148.912109375, 'global_p95': 255.9427734375, 'severity_ratio': 0.5818179875719899}]}```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# reuse existing dataframes if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure types
for df in (df_trace, df_log):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')

pods = ['productcatalogservice-0','productcatalogservice-1','productcatalogservice-2','productcatalogservice2-0',
        'currencyservice-0','currencyservice-2','currencyservice2-0']

start_ts = 1647797400
end_ts = 1647799200

# -------- Trace extraction ----------
mask_trace = df_trace['cmdb_id'].isin(pods) & (df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)
df_trace_win = df_trace.loc[mask_trace].copy()

# Group and compute aggregates
def trace_group_agg(grp):
    trace_name = grp.name[1] if isinstance(grp.name, tuple) else grp.name
    points = int(len(grp))
    mean_v = float(grp['value'].mean()) if not grp['value'].dropna().empty else None
    max_v = float(grp['value'].max()) if not grp['value'].dropna().empty else None
    errors_count = None
    sum_row_count = None
    if 'error_rate' in trace_name:
        errors_count = int((grp['value'] > 0).sum())
    if trace_name.endswith('row_count'):
        # sum of values (row_count)
        sum_row_count = float(grp['value'].sum()) if not grp['value'].dropna().empty else 0.0
    return pd.Series({
        'points_in_window': points,
        'mean_value': mean_v,
        'max_value': max_v,
        'errors_count': errors_count,
        'sum_row_count': sum_row_count
    })

if not df_trace_win.empty:
    trace_grp = df_trace_win.groupby(['cmdb_id', 'trace_name'], sort=True).apply(trace_group_agg).reset_index()
else:
    trace_grp = pd.DataFrame(columns=['cmdb_id','trace_name','points_in_window','mean_value','max_value','errors_count','sum_row_count'])

# For sorting, treat None as 0 for errors_count
trace_grp['_errors_sort'] = trace_grp['errors_count'].fillna(0).astype(int)
trace_sorted = trace_grp.sort_values(by=['_errors_sort','max_value'], ascending=[False, False]).drop(columns=['_errors_sort']).head(20)

# Convert NaNs to None for JSON-serializable output
trace_sorted = trace_sorted.where(pd.notnull(trace_sorted), None)

# -------- Log extraction ----------
mask_log = df_log['cmdb_id'].isin(pods) & (df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)
df_log_win = df_log.loc[mask_log].copy()

log_records = []
for pod in pods:
    sub = df_log_win.loc[df_log_win['cmdb_id'] == pod]
    points_in_window = int(len(sub))
    total_row_count = float(sub.loc[sub['log_name'] == 'log.row_count', 'value'].sum()) if not sub.empty else 0.0
    total_error_count = float(sub.loc[sub['log_name'] == 'log.error_count', 'value'].sum()) if not sub.empty else 0.0
    # earliest error timestamp where log_name=='log.error_count' and value>0
    err_rows = sub.loc[(sub['log_name'] == 'log.error_count') & (sub['value'] > 0)]
    if not err_rows.empty:
        earliest_ts = int(err_rows['timestamp'].min())
        earliest_iso = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).isoformat()
    else:
        earliest_iso = None
    log_records.append({
        'cmdb_id': pod,
        'points_in_window': points_in_window,
        'total_row_count': total_row_count,
        'total_error_count': total_error_count,
        'earliest_error_timestamp': earliest_iso
    })

log_df = pd.DataFrame(log_records)

# Build JSON-serializable result
trace_summary = trace_sorted[['cmdb_id','trace_name','points_in_window','mean_value','max_value','errors_count','sum_row_count']].to_dict(orient='records')
log_summary = log_df.to_dict(orient='records')

result = {
    'trace_summary': trace_summary,
    'log_summary': log_summary
}

result

```
Out[6]:
```
Summary (plain English):

Trace findings (window 2022-03-21 01:30–02:00 UTC+8)
- productcatalogservice2-0 shows very high inbound row_count from frontend2-0 (mean ≈ 180 req/min, sum ≈ 5,411 over the window, 30 points), and productcatalogservice-0/1/2 each show ~59 req/min from frontend pods (sum ≈ 1,768–1,776). These align with the network packet anomalies seen earlier for productcatalogservice pods.
- currencyservice pods show lower trace volume (mean ~37 req/min from frontend sources, sum ≈ 1,130), but earlier CPU/memory anomalies were observed on currencyservice-0/-2.
- No trace error_rate entries appeared in the top results (errors_count is null for the listed trace_names), so the trace aggregates returned are row_count dominated (no obvious trace-level error spikes in these top rows).

Log findings (same window)
- productcatalogservice-0/1/2: each has 60 log points and large total_row_count (~10,334–10,364) and total_error_count = 0. No error log timestamps found.
- productcatalogservice2-0: zero log rows in log.csv for the window (points_in_window = 0, total_row_count = 0). This pod had high trace traffic and network anomalies but no matching log entries — investigate whether it uses different logging pipeline, different cmdb_id, or missed ingestion.
- currencyservice-0 and currencyservice-2: 60 and 62 log points respectively, with large total_row_count (~7,492–7,499) and zero error_count.
- currencyservice2-0: zero log rows in log.csv (points_in_window = 0).

Interpretation and recommended next steps
- Productcatalogservice network spike corroboration: productcatalogservice2-0 (and productcatalogservice-0/1/2) show high request volumes in traces and corresponding packet transmit/receive anomalies. Investigate traffic sources (frontend2-0), recent deploys, routing rules, and Istio/mesh metrics for these pods.
- Missing logs: productcatalogservice2-0 and currencyservice2-0 have no log rows in log.csv for the window while traces/metrics show activity/anomalies — check log collection, logging config, or cmdb_id mismatches for these pods.
- Currencyservice CPU/memory: CPU and slight memory exceedances on currencyservice-0/-2 occurred despite moderate trace volume — inspect application-level logs, GC/pause events, and request latencies for currencyservice; consider scaling or throttling investigations.
- Correlate traces, mesh metrics, and node/container resource metrics for the highlighted pods to identify whether spikes are traffic-driven or due to resource contention.

If you want, I can pull trace duration/error_rate rows (if present) and Istio/mesh metrics for these pods in the same window, or list exact timestamps of the trace and network peaks for deeper correlation.

The original code execution output of IPython Kernel is also provided below for reference:

{'trace_summary': [{'cmdb_id': 'productcatalogservice2-0', 'trace_name': 'trace.from_frontend2-0.row_count', 'points_in_window': 30.0, 'mean_value': 180.36666666666667, 'max_value': 228.0, 'errors_count': nan, 'sum_row_count': 5411.0}, {'cmdb_id': 'currencyservice2-0', 'trace_name': 'trace.from_frontend2-0.row_count', 'points_in_window': 30.0, 'mean_value': 113.23333333333333, 'max_value': 160.0, 'errors_count': nan, 'sum_row_count': 3397.0}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30.0, 'mean_value': 58.96666666666667, 'max_value': 74.0, 'errors_count': nan, 'sum_row_count': 1769.0}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30.0, 'mean_value': 58.96666666666667, 'max_value': 74.0, 'errors_count': nan, 'sum_row_count': 1769.0}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30.0, 'mean_value': 58.93333333333333, 'max_value': 74.0, 'errors_count': nan, 'sum_row_count': 1768.0}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30.0, 'mean_value': 59.166666666666664, 'max_value': 71.0, 'errors_count': nan, 'sum_row_count': 1775.0}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30.0, 'mean_value': 59.2, 'max_value': 70.0, 'errors_count': nan, 'sum_row_count': 1776.0}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30.0, 'mean_value': 59.166666666666664, 'max_value': 70.0, 'errors_count': nan, 'sum_row_count': 1775.0}, {'cmdb_id': 'currencyservice-0', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30.0, 'mean_value': 37.7, 'max_value': 49.0, 'errors_count': nan, 'sum_row_count': 1131.0}, {'cmdb_id': 'currencyservice-0', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30.0, 'mean_value': 37.666666666666664, 'max_value': 49.0, 'errors_count': nan, 'sum_row_count': 1130.0}, {'cmdb_id': 'currencyservice-2', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30.0, 'mean_value': 37.7, 'max_value': 49.0, 'errors_count': nan, 'sum_row_count': 1131.0}, {'cmdb_id': 'currencyservice-2', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30.0, 'mean_value': 37.63333333333333, 'max_value': 49.0, 'errors_count': nan, 'sum_row_count': 1129.0}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30.0, 'mean_value': 20.233333333333334, 'max_value': 30.0, 'errors_count': nan, 'sum_row_count': 607.0}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30.0, 'mean_value': 20.266666666666666, 'max_value': 30.0, 'errors_count': nan, 'sum_row_count': 608.0}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30.0, 'mean_value': 20.233333333333334, 'max_value': 30.0, 'errors_count': nan, 'sum_row_count': 607.0}, {'cmdb_id': 'currencyservice-0', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30.0, 'mean_value': 12.866666666666667, 'max_value': 20.0, 'errors_count': nan, 'sum_row_count': 386.0}, {'cmdb_id': 'currencyservice-2', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30.0, 'mean_value': 12.9, 'max_value': 20.0, 'errors_count': nan, 'sum_row_count': 387.0}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_recommendationservice2-0.row_count', 'points_in_window': 30.0, 'mean_value': 9.366666666666667, 'max_value': 12.0, 'errors_count': nan, 'sum_row_count': 281.0}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_recommendationservice2-0.row_count', 'points_in_window': 30.0, 'mean_value': 9.333333333333334, 'max_value': 12.0, 'errors_count': nan, 'sum_row_count': 280.0}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_recommendationservice2-0.row_count', 'points_in_window': 30.0, 'mean_value': 9.366666666666667, 'max_value': 12.0, 'errors_count': nan, 'sum_row_count': 281.0}], 'log_summary': [{'cmdb_id': 'productcatalogservice-0', 'points_in_window': 60, 'total_row_count': 10334.0, 'total_error_count': 0.0, 'earliest_error_timestamp': None}, {'cmdb_id': 'productcatalogservice-1', 'points_in_window': 60, 'total_row_count': 10355.0, 'total_error_count': 0.0, 'earliest_error_timestamp': None}, {'cmdb_id': 'productcatalogservice-2', 'points_in_window': 60, 'total_row_count': 10364.0, 'total_error_count': 0.0, 'earliest_error_timestamp': None}, {'cmdb_id': 'productcatalogservice2-0', 'points_in_window': 0, 'total_row_count': 0.0, 'total_error_count': 0.0, 'earliest_error_timestamp': None}, {'cmdb_id': 'currencyservice-0', 'points_in_window': 60, 'total_row_count': 7492.0, 'total_error_count': 0.0, 'earliest_error_timestamp': None}, {'cmdb_id': 'currencyservice-2', 'points_in_window': 62, 'total_row_count': 7499.0, 'total_error_count': 0.0, 'earliest_error_timestamp': None}, {'cmdb_id': 'currencyservice2-0', 'points_in_window': 0, 'total_row_count': 0.0, 'total_error_count': 0.0, 'earliest_error_timestamp': None}]}```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import re
import pytz

# reuse df_trace from previous state
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure types
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['cmdb_id'] = df_trace['cmdb_id'].astype(str)
df_trace['trace_name'] = df_trace['trace_name'].astype(str)

pods = ['productcatalogservice-0','productcatalogservice-1','productcatalogservice-2','productcatalogservice2-0',
        'currencyservice-0','currencyservice-2']
start_ts = 1647797400
end_ts = 1647799200

patterns = ['trace.self.', 'trace.from_frontend-0.', 'trace.from_frontend-1.', 'trace.from_frontend-2.', 'trace.from_frontend2-0.']
pattern_regex = '|'.join(re.escape(p) for p in patterns)

# Filter rows
mask = df_trace['cmdb_id'].isin(pods) & df_trace['timestamp'].between(start_ts, end_ts) & df_trace['trace_name'].str.contains(pattern_regex, na=False)
df_win = df_trace.loc[mask].copy()

tz = pytz.timezone('Asia/Shanghai')

# Group and compute aggregates
def agg_group(grp):
    trace_name = grp.name[1] if isinstance(grp.name, tuple) else grp.name
    points = int(len(grp))
    mean_v = float(grp['value'].mean()) if not grp['value'].dropna().empty else None
    max_v = float(grp['value'].max()) if not grp['value'].dropna().empty else None
    if max_v is not None and not np.isnan(max_v):
        ts_max = grp.loc[grp['value'] == max_v, 'timestamp'].min()
        earliest_ts_max = pd.to_datetime(int(ts_max), unit='s', utc=True).tz_convert(tz).isoformat() if pd.notnull(ts_max) else None
    else:
        earliest_ts_max = None
    errors_count = int((grp['value'] > 0).sum()) if 'error_rate' in trace_name else None
    sum_row_count = float(grp['value'].sum()) if trace_name.endswith('row_count') else None
    return pd.Series({
        'points_in_window': points,
        'mean_value': mean_v,
        'max_value': max_v,
        'earliest_timestamp_of_max': earliest_ts_max,
        'errors_count': errors_count,
        'sum_row_count': sum_row_count
    })

if not df_win.empty:
    df_summary = df_win.groupby(['cmdb_id', 'trace_name'], sort=True).apply(agg_group).reset_index()
else:
    df_summary = pd.DataFrame(columns=['cmdb_id','trace_name','points_in_window','mean_value','max_value','earliest_timestamp_of_max','errors_count','sum_row_count'])

# Sorting: treat None as 0 for errors_count, and -inf for max_value if missing
df_summary['_errors_sort'] = df_summary['errors_count'].fillna(0).astype(int)
df_summary['_max_sort'] = df_summary['max_value'].fillna(-np.inf)
df_sorted = df_summary.sort_values(by=['_errors_sort', '_max_sort'], ascending=[False, False]).drop(columns=['_errors_sort','_max_sort'])

# Limit to top 30 and make JSON-serializable (NaN->None)
df_top = df_sorted.head(30).where(pd.notnull(df_sorted), None)

result = {
    'trace_detailed_summary': df_top[['cmdb_id','trace_name','points_in_window','mean_value','max_value','earliest_timestamp_of_max','errors_count','sum_row_count']].to_dict(orient='records')
}

result

```
Out[7]:
```
Summary (plain English):

- High request volume to productcatalogservice2-0:
  - trace.from_frontend2-0.row_count — mean ≈ 180 req/min, sum ≈ 5,411 over the window, 30 points.
  - Peak = 228 requests at 2022-03-21T01:40:00+08:00.
  - This aligns with earlier network packet anomalies for productcatalogservice2-0.

- productcatalogservice-0/1/2 traffic:
  - Each sees ~59 req/min from frontend-0 and frontend-1 (sum ≈ 1,768–1,776 per pod across the window).
  - Peaks (max row_count ~70–74) occur around 01:38–01:50 (UTC+8).
  - Additional modest traffic from frontend-2 (mean ≈ 20 req/min) and recommendationservice2-0 (≈9 req/min).

- currencyservice pods:
  - currencyservice-0 and -2: ~37 req/min from frontend sources (sum ≈ 1,130 each over the window).
  - Their max row_count values occur around 01:30–01:39 and 01:50 (UTC+8).

- Latency / duration traces:
  - trace.*.duration_p95 and duration_mean for productcatalogservice and productcatalogservice2-0 are very small (~0.005–0.006s p95), so request latency does not appear elevated in traces during the window.

- Errors:
  - No trace.error_rate spikes appear in the returned records (errors_count is not reported), so traces do not show notable error-rate increases for these trace names.

- Overall interpretation and recommended next steps:
  - The primary anomaly is traffic-driven: productcatalogservice2-0 experienced a sustained, high inbound load from frontend2-0 (peak at ~01:40), and productcatalogservice-0/1/2 also had elevated request volumes with peaks near 01:38–01:50.
  - Because durations remain low, the issue looks like increased traffic (not higher per-request latency).
  - Action items: investigate why frontend2-0 directed so much traffic to productcatalogservice2-0 (routing/traffic split, recent deploys, or spike in client requests), check productcatalogservice2-0 logs and logging pipeline (note earlier there were no log rows for productcatalogservice2-0 in the log summary), and correlate with the network packet anomalies and node/container metrics already flagged.
  - If you want, I can extract the exact per-minute trace and metric timestamps around the peaks for tighter correlation.

The original code execution output of IPython Kernel is also provided below for reference:

{'trace_detailed_summary': [{'cmdb_id': 'productcatalogservice2-0', 'trace_name': 'trace.from_frontend2-0.row_count', 'points_in_window': 30, 'mean_value': 180.36666666666667, 'max_value': 228.0, 'earliest_timestamp_of_max': '2022-03-21T01:40:00+08:00', 'errors_count': nan, 'sum_row_count': 5411.0}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30, 'mean_value': 58.96666666666667, 'max_value': 74.0, 'earliest_timestamp_of_max': '2022-03-21T01:38:00+08:00', 'errors_count': nan, 'sum_row_count': 1769.0}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30, 'mean_value': 58.96666666666667, 'max_value': 74.0, 'earliest_timestamp_of_max': '2022-03-21T01:38:00+08:00', 'errors_count': nan, 'sum_row_count': 1769.0}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30, 'mean_value': 58.93333333333333, 'max_value': 74.0, 'earliest_timestamp_of_max': '2022-03-21T01:38:00+08:00', 'errors_count': nan, 'sum_row_count': 1768.0}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30, 'mean_value': 59.166666666666664, 'max_value': 71.0, 'earliest_timestamp_of_max': '2022-03-21T01:50:00+08:00', 'errors_count': nan, 'sum_row_count': 1775.0}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30, 'mean_value': 59.2, 'max_value': 70.0, 'earliest_timestamp_of_max': '2022-03-21T01:50:00+08:00', 'errors_count': nan, 'sum_row_count': 1776.0}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30, 'mean_value': 59.166666666666664, 'max_value': 70.0, 'earliest_timestamp_of_max': '2022-03-21T01:50:00+08:00', 'errors_count': nan, 'sum_row_count': 1775.0}, {'cmdb_id': 'currencyservice-0', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30, 'mean_value': 37.7, 'max_value': 49.0, 'earliest_timestamp_of_max': '2022-03-21T01:30:00+08:00', 'errors_count': nan, 'sum_row_count': 1131.0}, {'cmdb_id': 'currencyservice-0', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30, 'mean_value': 37.666666666666664, 'max_value': 49.0, 'earliest_timestamp_of_max': '2022-03-21T01:37:00+08:00', 'errors_count': nan, 'sum_row_count': 1130.0}, {'cmdb_id': 'currencyservice-2', 'trace_name': 'trace.from_frontend-0.row_count', 'points_in_window': 30, 'mean_value': 37.7, 'max_value': 49.0, 'earliest_timestamp_of_max': '2022-03-21T01:30:00+08:00', 'errors_count': nan, 'sum_row_count': 1131.0}, {'cmdb_id': 'currencyservice-2', 'trace_name': 'trace.from_frontend-1.row_count', 'points_in_window': 30, 'mean_value': 37.63333333333333, 'max_value': 49.0, 'earliest_timestamp_of_max': '2022-03-21T01:39:00+08:00', 'errors_count': nan, 'sum_row_count': 1129.0}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30, 'mean_value': 20.233333333333334, 'max_value': 30.0, 'earliest_timestamp_of_max': '2022-03-21T01:38:00+08:00', 'errors_count': nan, 'sum_row_count': 607.0}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30, 'mean_value': 20.266666666666666, 'max_value': 30.0, 'earliest_timestamp_of_max': '2022-03-21T01:38:00+08:00', 'errors_count': nan, 'sum_row_count': 608.0}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30, 'mean_value': 20.233333333333334, 'max_value': 30.0, 'earliest_timestamp_of_max': '2022-03-21T01:38:00+08:00', 'errors_count': nan, 'sum_row_count': 607.0}, {'cmdb_id': 'currencyservice-0', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30, 'mean_value': 12.866666666666667, 'max_value': 20.0, 'earliest_timestamp_of_max': '2022-03-21T01:31:00+08:00', 'errors_count': nan, 'sum_row_count': 386.0}, {'cmdb_id': 'currencyservice-2', 'trace_name': 'trace.from_frontend-2.row_count', 'points_in_window': 30, 'mean_value': 12.9, 'max_value': 20.0, 'earliest_timestamp_of_max': '2022-03-21T01:50:00+08:00', 'errors_count': nan, 'sum_row_count': 387.0}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-2.duration_p95', 'points_in_window': 30, 'mean_value': 0.005757266666666666, 'max_value': 0.007549, 'earliest_timestamp_of_max': '2022-03-21T01:34:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-2.duration_p95', 'points_in_window': 30, 'mean_value': 0.005886466666666667, 'max_value': 0.007133, 'earliest_timestamp_of_max': '2022-03-21T01:30:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-1.duration_p95', 'points_in_window': 30, 'mean_value': 0.0060845333333333345, 'max_value': 0.007086, 'earliest_timestamp_of_max': '2022-03-21T01:38:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-1.duration_p95', 'points_in_window': 30, 'mean_value': 0.006030066666666666, 'max_value': 0.006793, 'earliest_timestamp_of_max': '2022-03-21T01:30:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-0.duration_p95', 'points_in_window': 30, 'mean_value': 0.0060398000000000006, 'max_value': 0.006751, 'earliest_timestamp_of_max': '2022-03-21T01:55:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-0.duration_p95', 'points_in_window': 30, 'mean_value': 0.005940333333333333, 'max_value': 0.00663, 'earliest_timestamp_of_max': '2022-03-21T01:45:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-1.duration_p95', 'points_in_window': 30, 'mean_value': 0.0059823666666666666, 'max_value': 0.006571, 'earliest_timestamp_of_max': '2022-03-21T01:30:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice2-0', 'trace_name': 'trace.from_frontend2-0.duration_p95', 'points_in_window': 30, 'mean_value': 0.006060266666666665, 'max_value': 0.00654, 'earliest_timestamp_of_max': '2022-03-21T01:45:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-0.duration_p95', 'points_in_window': 30, 'mean_value': 0.005988599999999999, 'max_value': 0.00648, 'earliest_timestamp_of_max': '2022-03-21T01:51:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-2.duration_p95', 'points_in_window': 30, 'mean_value': 0.005695633333333334, 'max_value': 0.006365, 'earliest_timestamp_of_max': '2022-03-21T01:53:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-1.duration_mean', 'points_in_window': 30, 'mean_value': 0.004822866666666666, 'max_value': 0.005387, 'earliest_timestamp_of_max': '2022-03-21T01:41:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-1', 'trace_name': 'trace.from_frontend-2.duration_mean', 'points_in_window': 30, 'mean_value': 0.004785866666666667, 'max_value': 0.005265, 'earliest_timestamp_of_max': '2022-03-21T01:45:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-2', 'trace_name': 'trace.from_frontend-2.duration_mean', 'points_in_window': 30, 'mean_value': 0.004759233333333333, 'max_value': 0.005262, 'earliest_timestamp_of_max': '2022-03-21T01:47:00+08:00', 'errors_count': nan, 'sum_row_count': nan}, {'cmdb_id': 'productcatalogservice-0', 'trace_name': 'trace.from_frontend-2.duration_mean', 'points_in_window': 30, 'mean_value': 0.0047628, 'max_value': 0.005246, 'earliest_timestamp_of_max': '2022-03-21T01:43:00+08:00', 'errors_count': nan, 'sum_row_count': nan}]}```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# reuse df_trace if available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure types
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['trace_name'] = df_trace['trace_name'].astype(str)
df_trace['cmdb_id'] = df_trace['cmdb_id'].astype(str)

# Window and filter terms
start_ts = 1647797400
end_ts = 1647799200
terms_regex = 'productcatalogservice|currencyservice'  # case-sensitive substring match

# Filter by time window and trace_name containing the terms
mask = df_trace['timestamp'].between(start_ts, end_ts) & df_trace['trace_name'].str.contains(terms_regex, na=False)
df_win = df_trace.loc[mask].copy()

# Group by trace_name and compute aggregates
tz = pytz.timezone('Asia/Shanghai')

if not df_win.empty:
    grp = df_win.groupby('trace_name', sort=True)
    summary_df = grp.agg(
        row_count=('trace_name', 'size'),
        earliest_ts=('timestamp', 'min'),
        sample_cmdb_ids=('cmdb_id', lambda s: sorted(s.dropna().unique())[:5])
    ).reset_index()
    # Convert earliest timestamp to ISO with UTC+8
    def to_iso(ts):
        if pd.isna(ts):
            return None
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()
    summary_df['earliest_timestamp'] = summary_df['earliest_ts'].apply(to_iso)
    summary_df = summary_df.drop(columns=['earliest_ts'])
    # Sort by row_count desc, then trace_name asc, limit to 50
    summary_df = summary_df.sort_values(['row_count','trace_name'], ascending=[False, True]).head(50)
    # Make JSON-serializable: replace any NaN with None
    summary_df = summary_df.where(pd.notnull(summary_df), None)
    result = {
        'total_trace_names': int(summary_df.shape[0]),
        'trace_names': summary_df[['trace_name','row_count','sample_cmdb_ids','earliest_timestamp']].to_dict(orient='records')
    }
else:
    result = {'total_trace_names': 0, 'trace_names': []}

result

```
Out[8]:
```
Findings (plain English):

- Scope: 32 distinct trace_name values reference "productcatalogservice" or "currencyservice" within the window 2022-03-21 01:30–02:00 (UTC+8).

- Major groups and density:
  - trace.to_productcatalogservice-2.* (duration_mean / duration_p95 / error_rate / row_count): 268 rows each; earliest appearance 01:30. Caller examples: checkoutservice-0/1/2 and frontend-0/1.
  - trace.to_productcatalogservice-0.* and trace.to_productcatalogservice-1.*: 266 rows each; earliest 01:30. Caller examples: checkoutservice-0/1/2 and frontend-0/1.
  - trace.to_productcatalogservice2-0.*: 55 rows each (earliest 01:30). Caller examples: checkoutservice2-0 and frontend2-0.
  - trace.to_currencyservice-0.*, trace.to_currencyservice-1.*, trace.to_currencyservice-2.*: ~160–161 rows each (earliest 01:30). Caller examples: checkoutservice-0/1/2 and frontend-0/1.
  - trace.to_currencyservice2-0.*: 55 rows each (earliest 01:30); caller examples: checkoutservice2-0 and frontend2-0.

- What this implies:
  - The dominated trace names are "to_*" metrics (calls INTO productcatalogservice and currencyservice), with consistent, repeated per-minute data across the window.
  - Callers are primarily checkoutservice pods and frontend pods (including frontend2-0 for the *2-0 variants).
  - All trace_names of interest first appear at the start of the window (01:30 UTC+8).
  - The presence of row_count and error_rate series for these targets indicates both traffic volume and error monitoring are available; however, counts shown are row_count-heavy and earliest timestamps align with window start.

- Suggested next steps:
  - Correlate the high row_count series (especially toward productcatalogservice2-0) with the metric/network anomalies already identified.
  - Inspect the error_rate series for any nonzero values (though these aggregates only show existence and counts here).
  - Focus on caller components (checkoutservice and frontend pods) to see why they generated increased traffic to productcatalogservice / currencyservice during the window.

The original code execution output of IPython Kernel is also provided below for reference:

{'total_trace_names': 32, 'trace_names': [{'trace_name': 'trace.to_productcatalogservice-2.duration_mean', 'row_count': 268, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-2.duration_p95', 'row_count': 268, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-2.error_rate', 'row_count': 268, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-2.row_count', 'row_count': 268, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-0.duration_mean', 'row_count': 266, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-0.duration_p95', 'row_count': 266, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-0.error_rate', 'row_count': 266, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-0.row_count', 'row_count': 266, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-1.duration_mean', 'row_count': 266, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-1.duration_p95', 'row_count': 266, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-1.error_rate', 'row_count': 266, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice-1.row_count', 'row_count': 266, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-1.duration_mean', 'row_count': 161, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-1.duration_p95', 'row_count': 161, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-1.error_rate', 'row_count': 161, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-1.row_count', 'row_count': 161, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-0.duration_mean', 'row_count': 160, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-0.duration_p95', 'row_count': 160, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-0.error_rate', 'row_count': 160, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-0.row_count', 'row_count': 160, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-2.duration_mean', 'row_count': 160, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-2.duration_p95', 'row_count': 160, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-2.error_rate', 'row_count': 160, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice-2.row_count', 'row_count': 160, 'sample_cmdb_ids': ['checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'frontend-0', 'frontend-1'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice2-0.duration_mean', 'row_count': 55, 'sample_cmdb_ids': ['checkoutservice2-0', 'frontend2-0'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice2-0.duration_p95', 'row_count': 55, 'sample_cmdb_ids': ['checkoutservice2-0', 'frontend2-0'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice2-0.error_rate', 'row_count': 55, 'sample_cmdb_ids': ['checkoutservice2-0', 'frontend2-0'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_currencyservice2-0.row_count', 'row_count': 55, 'sample_cmdb_ids': ['checkoutservice2-0', 'frontend2-0'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice2-0.duration_mean', 'row_count': 55, 'sample_cmdb_ids': ['checkoutservice2-0', 'frontend2-0'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice2-0.duration_p95', 'row_count': 55, 'sample_cmdb_ids': ['checkoutservice2-0', 'frontend2-0'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice2-0.error_rate', 'row_count': 55, 'sample_cmdb_ids': ['checkoutservice2-0', 'frontend2-0'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}, {'trace_name': 'trace.to_productcatalogservice2-0.row_count', 'row_count': 55, 'sample_cmdb_ids': ['checkoutservice2-0', 'frontend2-0'], 'earliest_timestamp': '2022-03-21T01:30:00+08:00'}]}```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# reuse df_trace if available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure types and columns
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['trace_name'] = df_trace['trace_name'].astype(str)
df_trace['cmdb_id'] = df_trace['cmdb_id'].astype(str)

start_ts = 1647797400
end_ts = 1647799200
tz = pytz.timezone('Asia/Shanghai')

# Window filter
df_win = df_trace[df_trace['timestamp'].between(start_ts, end_ts)].copy()

results = []

# 1) trace_name contains 'to_currencyservice' AND cmdb_id starts with productcatalogservice or productcatalogservice2
mask1 = df_win['trace_name'].str.contains('to_currencyservice', na=False) & \
        df_win['cmdb_id'].str.startswith(('productcatalogservice','productcatalogservice2'))
df1 = df_win.loc[mask1]

if not df1.empty:
    grp1 = df1.groupby('trace_name', sort=True)
    recs1 = []
    for name, g in grp1:
        row_count = int(len(g))
        sample_cmdb_ids = sorted(g['cmdb_id'].dropna().unique())[:10]
        earliest_ts = int(g['timestamp'].min())
        earliest_iso = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).isoformat()
        recs1.append({
            'direction': 'productcatalogservice->currencyservice',
            'trace_name': name,
            'row_count': row_count,
            'sample_cmdb_ids': sample_cmdb_ids,
            'earliest_timestamp': earliest_iso
        })
    # sort by row_count desc, then trace_name
    recs1 = sorted(recs1, key=lambda x: (-x['row_count'], x['trace_name']))
    results.extend(recs1)

# 2) trace_name contains 'to_productcatalogservice' AND cmdb_id starts with currencyservice or currencyservice2
mask2 = df_win['trace_name'].str.contains('to_productcatalogservice', na=False) & \
        df_win['cmdb_id'].str.startswith(('currencyservice','currencyservice2'))
df2 = df_win.loc[mask2]

if not df2.empty:
    grp2 = df2.groupby('trace_name', sort=True)
    recs2 = []
    for name, g in grp2:
        row_count = int(len(g))
        sample_cmdb_ids = sorted(g['cmdb_id'].dropna().unique())[:10]
        earliest_ts = int(g['timestamp'].min())
        earliest_iso = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).isoformat()
        recs2.append({
            'direction': 'currencyservice->productcatalogservice',
            'trace_name': name,
            'row_count': row_count,
            'sample_cmdb_ids': sample_cmdb_ids,
            'earliest_timestamp': earliest_iso
        })
    recs2 = sorted(recs2, key=lambda x: (-x['row_count'], x['trace_name']))
    results.extend(recs2)

# Limit total returned records to at most 50
results_limited = results[:50]

result = {
    'total_returned': len(results_limited),
    'records': results_limited
}

result

```
Out[9]:
```
Result: No matching cross-service trace entries found in the window.

Plain English summary:
- I searched trace rows in 2022-03-21 01:30–02:00 (UTC+8) for:
  1) traces that contain "to_currencyservice" and originate from cmdb_id starting with "productcatalogservice" or "productcatalogservice2", and
  2) traces that contain "to_productcatalogservice" and originate from cmdb_id starting with "currencyservice" or "currencyservice2".
- Outcome: zero records matched either condition in that time window.

Interpretation and next steps:
- There are no direct productcatalogservice→currencyservice or currencyservice→productcatalogservice "to_*" traces recorded in this window (at least not with cmdb_id naming that starts with those service prefixes).
- Earlier queries showed that most "to_productcatalogservice" and "to_currencyservice" traces were emitted by frontend and checkoutservice callers (checkoutservice-*, frontend-*), not by productcatalogservice or currencyservice themselves.
- If you want to find cross-service calls in the other direction, try:
  - Searching for trace names with "from_productcatalogservice" / "from_currencyservice" or
  - Looking for traces where cmdb_id is productcatalogservice* or currencyservice* and trace_name contains "trace.to_" (caller/target naming can vary), or
  - Expanding the time window or doing a case-insensitive / partial-name scan to catch naming variants.

The original code execution output of IPython Kernel is also provided below for reference:

{'total_returned': 0, 'records': []}```
```